In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/data-synthetic-noise/test.tsv
/kaggle/input/data-synthetic-noise/dev.tsv
/kaggle/input/data-synthetic-noise/train.tsv


In [2]:
df = pd.read_csv('/kaggle/input/data-synthetic-noise/train.tsv')
df.head()

,input\ttarget\tsource_rule\tfreq\tedit_count\tseed\tfreq_bucket
0,intranel\tintranet\tocr_confusion\t6.76e-07\t1...
1,evrasiático\teurasiático\thistorical\t0.0\t1\t...
2,coñsumido\tconsumido\tocr_confusion\t3.09e-06\...
3,posadaas\tposadas\ttypo\t3.39e-06\t1\t42\t4
4,aqlegamar\talegamar\ttypo\t0.0\t1\t42\t0


In [3]:
# Spanish Post-OCR + 16th-17th Century Text Normalization with LoRA Fine-tuning
# Single Kaggle notebook for word-level Spanish text normalization using T5 + LoRA/PEFT
# Optimized for Kaggle GPU (12-16GB VRAM) with lexicon rescoring

# Cell 1: Install & Imports
# Runtime: ~2-3 minutes, Memory: minimal

!pip install transformers==4.35.0 datasets==2.14.0 accelerate==0.24.0 peft==0.6.0 evaluate==0.4.0
!pip install tokenizers==0.15.0 sentencepiece==0.1.99 rapidfuzz==3.5.0 pandas==1.5.3 tqdm==4.66.0
# Try bitsandbytes with fallback

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 123.1/123.1 kB 841.6 kB/s eta 0:00:00a 0:00:01
INFO: pip is looking at multiple versions of multiprocess to determine which version is compatible with other requirements. This could take a while.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.9/7.9 MB 17.0 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 492.2/492.2 kB 20.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 261.0/261.0 kB 16.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 134.9/134.9 kB 9.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 81.4/81.4 kB 4.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 115.3/115.3 kB 6.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.8/3.8 MB 81.5 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 295.0/295.0 kB 15.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 4.4 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━

In [4]:
# pip uninstall -y bitsandbytes

In [5]:
try:
    !pip install bitsandbytes

    QUANTIZATION_AVAILABLE = True
    print("✓ bitsandbytes installed - quantization available")
except:
    QUANTIZATION_AVAILABLE = False
    print("⚠ bitsandbytes failed - training without quantization")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.3/61.3 MB 21.5 MB/s eta 0:00:0000:0100:01
✓ bitsandbytes installed - quantization available


In [6]:
# pip install --upgrade --force-reinstall bitsandbytes


In [7]:
# pip install --upgrade --force-reinstall transformers peft accelerate


In [8]:
pip install rapidfuzz

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 6.6 MB/s eta 0:00:0000:0100:010m
Note: you may need to restart the kernel to use updated packages.


In [9]:
import os
import json
import random
import numpy as np
import pandas as pd
from tqdm.auto import tqdm
import torch
from transformers import (
    AutoTokenizer, AutoModelForSeq2SeqLM, Seq2SeqTrainer, Seq2SeqTrainingArguments,
    DataCollatorForSeq2Seq, EarlyStoppingCallback
)
from datasets import Dataset
from peft import LoraConfig, get_peft_model, TaskType, PeftModel
from evaluate import load
import rapidfuzz.process
import rapidfuzz.distance
import unicodedata
from accelerate import Accelerator

/usr/local/lib/python3.11/dist-packages/transformers/utils/generic.py:441: FutureWarning: `torch.utils._pytree._register_pytree_node` is deprecated. Please use `torch.utils._pytree.register_pytree_node` instead.
  _torch_pytree._register_pytree_node(
/usr/local/lib/python3.11/dist-packages/transformers/utils/generic.py:309: FutureWarning: `torch.utils._pytree._register_pytree_node` is deprecated. Please use `torch.utils._pytree.register_pytree_node` instead.
  _torch_pytree._register_pytree_node(
2025-08-28 05:19:11.075694: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1756358351.284149      36 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1756358351.341958      36 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to registe

In [10]:
pip install evaluate

Note: you may need to restart the kernel to use updated packages.


In [10]:
import os
import json
import random
import numpy as np
import pandas as pd
from tqdm.auto import tqdm
import torch
from transformers import (
    AutoTokenizer, AutoModelForSeq2SeqLM, Seq2SeqTrainer, Seq2SeqTrainingArguments,
    DataCollatorForSeq2Seq, TrainerCallback
)
from datasets import Dataset
from peft import LoraConfig, get_peft_model, TaskType, PeftModel
import rapidfuzz.distance
import unicodedata
import warnings
warnings.filterwarnings("ignore")

def seed_everything(seed=42):
    """Set seeds for reproducibility"""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)

seed_everything(42)
print("Environment setup complete")

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

def load_tsv_data(filepath):
    """Load TSV data"""
    try:
        df = pd.read_csv(filepath, sep='\t', encoding='utf-8')
        required_cols = ['input', 'target']
        
        missing_required = [col for col in required_cols if col not in df.columns]
        if missing_required:
            print(f"Missing required columns in {filepath}: {missing_required}")
            return pd.DataFrame()
        
        return df
    except Exception as e:
        print(f"Error loading {filepath}: {e}")
        return pd.DataFrame()

# Load data
try:
    train_df = load_tsv_data('/kaggle/input/data-synthetic-noise/train.tsv')
    test_df = load_tsv_data('/kaggle/input/data-synthetic-noise/test.tsv')
    
    if len(train_df) == 0:
        print("No training data found")
        exit(1)
        
except FileNotFoundError:
    print("Data files not found")
    exit(1)

print(f"Training samples: {len(train_df)}")
print(f"Test samples: {len(test_df)}")

def clean_text(text):
    """Clean text"""
    if pd.isna(text):
        return ""
    text = unicodedata.normalize('NFC', str(text))
    allowed_chars = set('abcdefghijklmnopqrstuvwxyzABCDEFGHIJKLMNOPQRSTUVWXYZ'
                       'áéíóúüñÁÉÍÓÚÜÑàèìòùÀÈÌÒÙ'
                       '0123456789 .,;:!?¿¡()[]{}"-\'')
    text = ''.join(c for c in text if c in allowed_chars)
    return text.strip()

def prepare_dataset(df, name, max_samples=5000):
    """Prepare dataset with better filtering"""
    print(f"\nPreparing {name} dataset...")
    
    # Clean text
    df['input'] = df['input'].apply(clean_text)
    df['target'] = df['target'].apply(clean_text)
    
    # Remove empty entries
    df = df[(df['input'].str.len() > 0) & (df['target'].str.len() > 0)]
    df = df[df['input'].str.len() <= 50]  # Shorter max length
    df = df[df['target'].str.len() <= 50]
    
    # CRITICAL FIX: Remove most identity pairs that cause training collapse
    identity_mask = df['input'] == df['target']
    identity_count = identity_mask.sum()
    non_identity_count = (~identity_mask).sum()
    
    print(f"  Identity pairs: {identity_count}")
    print(f"  Non-identity pairs: {non_identity_count}")
    
    # Keep only 10% of identity pairs to prevent collapse
    if identity_count > 100:
        identity_samples = df[identity_mask].sample(n=min(100, identity_count), random_state=42)
        non_identity_samples = df[~identity_mask]
        df = pd.concat([non_identity_samples, identity_samples], ignore_index=True)
        print(f"  Reduced identity pairs to {len(identity_samples)}")
    
    # Sample if too large
    if len(df) > max_samples:
        df = df.sample(n=max_samples, random_state=42)
        print(f"  Sampled {max_samples} examples")
    
    print(f"  Final size: {len(df)}")
    
    # Show edit distance distribution
    if len(df) > 0:
        edit_distances = df.apply(lambda row: rapidfuzz.distance.Levenshtein.distance(
            row['input'], row['target']), axis=1)
        print(f"  Edit distance - mean: {edit_distances.mean():.2f}, max: {edit_distances.max()}")
        
        # Show sample data
        print(f"  Sample pairs:")
        for i in range(min(3, len(df))):
            print(f"    '{df.iloc[i]['input']}' -> '{df.iloc[i]['target']}'")
    
    return df.reset_index(drop=True)

# Prepare datasets
train_df = prepare_dataset(train_df, "train", max_samples=5000)
test_df = prepare_dataset(test_df, "test", max_samples=1000)

# # Model setup
# MODEL_NAME = "google/byt5-small"
# print(f"\nUsing model: {MODEL_NAME}")

# try:
#     tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
#     print(f"Tokenizer loaded, vocab size: {tokenizer.vocab_size}")
# except Exception as e:
#     print(f"Error loading tokenizer: {e}")
#     exit(1)
MODEL_NAME = "jorgeortizfuentes/spanish-spellchecker-t5-base-wiki200000"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSeq2SeqLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float32,
)

# CRITICAL: Use simpler prefix and validate tokenization
PREFIX = "fix: "
MAX_LENGTH = 64

print(f"Testing tokenization with prefix '{PREFIX}':")
test_input = PREFIX + "h0la"
test_target = "hola"
input_tokens = tokenizer(test_input, max_length=MAX_LENGTH, truncation=True)
target_tokens = tokenizer(test_target, max_length=MAX_LENGTH, truncation=True)
print(f"  Input: '{test_input}' -> {len(input_tokens['input_ids'])} tokens")
print(f"  Target: '{test_target}' -> {len(target_tokens['input_ids'])} tokens")

def preprocess_function(examples):
    """Tokenize with validation"""
    inputs = [PREFIX + inp for inp in examples['input']]
    targets = examples['target']
    
    model_inputs = tokenizer(inputs, max_length=MAX_LENGTH, truncation=True, padding=False)
    labels = tokenizer(targets, max_length=MAX_LENGTH, truncation=True, padding=False)
    
    # Validation
    valid_indices = []
    for i, (inp_ids, lbl_ids) in enumerate(zip(model_inputs['input_ids'], labels['input_ids'])):
        if len(inp_ids) > 0 and len(lbl_ids) > 0:
            valid_indices.append(i)
    
    if len(valid_indices) != len(model_inputs['input_ids']):
        print(f"Filtered out {len(model_inputs['input_ids']) - len(valid_indices)} invalid samples")
    
    # Keep only valid samples
    model_inputs['input_ids'] = [model_inputs['input_ids'][i] for i in valid_indices]
    model_inputs['attention_mask'] = [model_inputs['attention_mask'][i] for i in valid_indices]
    model_inputs['labels'] = [labels['input_ids'][i] for i in valid_indices]
    
    return model_inputs

# Create datasets
print("\nCreating datasets...")
train_dataset = Dataset.from_pandas(train_df[['input', 'target']])
train_dataset = train_dataset.map(preprocess_function, batched=True, remove_columns=['input', 'target'])
print(f"Train dataset: {len(train_dataset)} samples")

test_dataset = Dataset.from_pandas(test_df[['input', 'target']])
test_dataset = test_dataset.map(preprocess_function, batched=True, remove_columns=['input', 'target'])
print(f"Test dataset: {len(test_dataset)} samples")

# Show sample tokenized data
print("\nSample tokenized data:")
for i in range(min(2, len(train_dataset))):
    sample = train_dataset[i]
    input_text = tokenizer.decode(sample['input_ids'], skip_special_tokens=True)
    label_text = tokenizer.decode(sample['labels'], skip_special_tokens=True)
    print(f"  Input: '{input_text}'")
    print(f"  Label: '{label_text}'")

# Load model
print("\nLoading model...")
try:
    model = AutoModelForSeq2SeqLM.from_pretrained(
        MODEL_NAME,
        torch_dtype=torch.float32,  # Use float32 for stability
    )
    print("Base model loaded")
except Exception as e:
    print(f"Error loading model: {e}")
    exit(1)

# VERY CONSERVATIVE LoRA config
lora_config = LoraConfig(
    r=4,                               # Very small rank
    lora_alpha=4,                      # Small alpha
    target_modules=["q", "v"],         # Only 2 modules
    lora_dropout=0.05,                 
    bias="none",                       
    task_type=TaskType.SEQ_2_SEQ_LM,   
)

print("Applying LoRA...")
model = get_peft_model(model, lora_config)

if torch.cuda.is_available():
    model = model.cuda()
    print("Model moved to CUDA")

# Enable training mode
model.train()
for name, param in model.named_parameters():
    if 'lora' in name.lower():
        param.requires_grad = True

trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
total_params = sum(p.numel() for p in model.parameters())
print(f"Trainable: {trainable_params:,} ({100*trainable_params/total_params:.3f}%)")

# Test forward pass
print("\nTesting model...")
try:
    sample = train_dataset[0]
    input_ids = torch.tensor([sample['input_ids']]).cuda() if torch.cuda.is_available() else torch.tensor([sample['input_ids']])
    labels = torch.tensor([sample['labels']]).cuda() if torch.cuda.is_available() else torch.tensor([sample['labels']])
    
    with torch.no_grad():
        outputs = model(input_ids=input_ids, labels=labels)
        print(f"Forward pass successful, loss: {outputs.loss:.4f}")
        
except Exception as e:
    print(f"Model test failed: {e}")
    exit(1)

# VERY CONSERVATIVE training settings
training_args = Seq2SeqTrainingArguments(
    output_dir="./results",
    overwrite_output_dir=True,
    evaluation_strategy="no",
    save_strategy="no",
    logging_steps=100,
    per_device_train_batch_size=1,      # Smallest possible batch
    gradient_accumulation_steps=16,     # Large accumulation
    learning_rate=1e-4,                 # Very small learning rate
    num_train_epochs=5,                 # Just 1 epoch
    warmup_steps=50,
    weight_decay=0.01,
    fp16=False,                         # No mixed precision
    dataloader_pin_memory=False,
    remove_unused_columns=True,
    report_to=[],
    disable_tqdm=False,
    dataloader_num_workers=0,
    max_grad_norm=0.1,                  # Very strong gradient clipping
    prediction_loss_only=True,
)

print(f"Training settings: batch_size=1, grad_accum=16, lr=1e-5")

# Data collator
data_collator = DataCollatorForSeq2Seq(
    tokenizer=tokenizer,
    model=model,
    padding=True,
    max_length=MAX_LENGTH,
)

# Loss monitoring callback
class LossMonitorCallback(TrainerCallback):
    def __init__(self):
        self.losses = []
        
    def on_log(self, args, state, control, model=None, logs=None, **kwargs):
        if logs and 'train_loss' in logs:
            loss = logs['train_loss']
            step = state.global_step
            self.losses.append((step, loss))
            
            print(f"Step {step}: loss = {loss:.6f}")
            
            # Stop if loss becomes zero or explodes
            if loss == 0.0:
                print("ZERO LOSS - stopping training!")
                control.should_training_stop = True
            elif loss > 100:
                print("EXPLODING LOSS - stopping training!")
                control.should_training_stop = True

# Create trainer
trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    tokenizer=tokenizer,
    data_collator=data_collator,
)

# Add callback
loss_monitor = LossMonitorCallback()
trainer.add_callback(loss_monitor)

print("\nStarting training...")
try:
    train_result = trainer.train()
    print(f"Training completed! Final loss: {train_result.training_loss:.6f}")
    
    # Show loss progression
    if loss_monitor.losses:
        print("\nLoss progression:")
        for step, loss in loss_monitor.losses:
            print(f"  Step {step}: {loss:.6f}")
            
except Exception as e:
    print(f"Training failed: {e}")
    import traceback
    traceback.print_exc()

print("\nTesting inference...")

def test_model(input_text):
    """Test model inference"""
    try:
        model.eval()
        
        full_input = PREFIX + input_text
        input_ids = tokenizer.encode(full_input, return_tensors="pt")
        if torch.cuda.is_available():
            input_ids = input_ids.cuda()
        
        with torch.no_grad():
            outputs = model.generate(
                input_ids=input_ids,
                max_length=MAX_LENGTH,
                num_beams=1,
                do_sample=False,
                pad_token_id=tokenizer.pad_token_id,
                eos_token_id=tokenizer.eos_token_id
            )
            
            prediction = tokenizer.decode(outputs[0], skip_special_tokens=True).strip()
            return prediction
            
    except Exception as e:
        return f"ERROR: {e}"

# Test cases
test_cases = ["hola", "h0la", "casa", "c4sa"]
print("\nInference test:")
for test_input in test_cases:
    prediction = test_model(test_input)
    print(f"  '{test_input}' -> '{prediction}'")

# Test on actual data samples
print("\nTesting on real data samples:")
for i in range(min(3, len(test_df))):
    input_word = test_df.iloc[i]['input']
    target_word = test_df.iloc[i]['target']
    prediction = test_model(input_word)
    
    correct = "✓" if prediction == target_word else "✗"
    print(f"  '{input_word}' -> '{target_word}' | Pred: '{prediction}' {correct}")

print("\nDiagnostic complete!")
print("If you still see repetitive/garbled output, the fundamental training setup needs revision.")

Environment setup complete
PyTorch version: 2.6.0+cu124
CUDA available: True
GPU: Tesla P100-PCIE-16GB
Training samples: 109112
Test samples: 13639

Preparing train dataset...
  Identity pairs: 13639
  Non-identity pairs: 95471
  Reduced identity pairs to 100
  Sampled 5000 examples
  Final size: 5000
  Edit distance - mean: 1.22, max: 5
  Sample pairs:
    'escupeen' -> 'escupen'
    'cardnoso' -> 'carnoso'
    'pertiostio' -> 'periostio'

Preparing test dataset...
  Identity pairs: 1705
  Non-identity pairs: 11934
  Reduced identity pairs to 100
  Sampled 1000 examples
  Final size: 1000
  Edit distance - mean: 1.23, max: 4
  Sample pairs:
    'oerebróvafcular' -> 'cerebrovascular'
    'mámuts' -> 'mamuts'
    'agropecurios' -> 'agropecuarios'

Using model: google/byt5-small


Tokenizer loaded, vocab size: 256
Testing tokenization with prefix 'fix: ':
  Input: 'fix: h0la' -> 10 tokens
  Target: 'hola' -> 5 tokens

Creating datasets...


Map:   0%|          | 0/5000 [00:00<?, ? examples/s]

Train dataset: 5000 samples


Map:   0%|          | 0/1000 [00:00<?, ? examples/s]

Test dataset: 1000 samples

Sample tokenized data:
  Input: 'fix: escupeen'
  Label: 'escupen'
  Input: 'fix: cardnoso'
  Label: 'carnoso'

Loading model...


Base model loaded
Applying LoRA...
Model moved to CUDA
Trainable: 296,960 (0.099%)

Testing model...
Forward pass successful, loss: 5.3972
Training settings: batch_size=1, grad_accum=16, lr=1e-5

Starting training...


Step,Training Loss
100,6.549900
200,5.415600
300,4.185500
400,3.409000
500,2.954600
600,2.622600
700,2.306300
800,2.047500
900,1.819100
1000,1.710700


Step 1560: loss = 2.661345
Training completed! Final loss: 2.661345

Loss progression:
  Step 1560: 2.661345

Testing inference...

Inference test:
  'hola' -> 'ola'
  'h0la' -> 'h0la'
  'casa' -> 'casa'
  'c4sa' -> 'c4sa'

Testing on real data samples:
  'oerebróvafcular' -> 'cerebrovascular' | Pred: 'oerebróvafcular' ✗
  'mámuts' -> 'mamuts' | Pred: 'mamuts' ✓
  'agropecurios' -> 'agropecuarios' | Pred: 'agropecurios' ✗

Diagnostic complete!
If you still see repetitive/garbled output, the fundamental training setup needs revision.


In [12]:
# import os
# import json
# import random
# import numpy as np
# import pandas as pd
# from tqdm.auto import tqdm
# import torch
# from transformers import (
#     AutoTokenizer, AutoModelForSeq2SeqLM, Seq2SeqTrainer, Seq2SeqTrainingArguments,
#     DataCollatorForSeq2Seq, EarlyStoppingCallback
# )
# from datasets import Dataset
# from peft import LoraConfig, get_peft_model, TaskType, PeftModel
# from evaluate import load
# import rapidfuzz.process
# import rapidfuzz.distance
# import unicodedata
# from accelerate import Accelerator
# import warnings
# warnings.filterwarnings("ignore")

# # Configure accelerate for single GPU
# os.environ['ACCELERATE_USE_FSDP'] = 'false'
# os.environ['ACCELERATE_USE_DEEPSPEED'] = 'false'

# def seed_everything(seed=42):
#     """Set seeds for reproducibility"""
#     random.seed(seed)
#     np.random.seed(seed)
#     torch.manual_seed(seed)
#     torch.cuda.manual_seed_all(seed)
#     os.environ['PYTHONHASHSEED'] = str(seed)

# seed_everything(42)
# print("✓ Environment setup complete")

# # Check quantization availability
# try:
#     from transformers import BitsAndBytesConfig
#     QUANTIZATION_AVAILABLE = True
# except ImportError:
#     QUANTIZATION_AVAILABLE = False

# print(f"Python version: {os.sys.version}")
# print(f"PyTorch version: {torch.__version__}")
# print(f"CUDA available: {torch.cuda.is_available()}")
# if torch.cuda.is_available():
#     print(f"GPU: {torch.cuda.get_device_name(0)}")
#     print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
#     print(f"Current GPU memory: {torch.cuda.memory_allocated() / 1e6:.1f} MB")
# print(f"Quantization available: {QUANTIZATION_AVAILABLE}")

# def load_tsv_data(filepath):
#     """Load TSV with expected columns: input, target, source_rule, freq, edit_count, seed"""
#     try:
#         df = pd.read_csv(filepath, sep='\t', encoding='utf-8')
#         required_cols = ['input', 'target']
#         optional_cols = ['source_rule', 'freq', 'edit_count', 'seed']
        
#         missing_required = [col for col in required_cols if col not in df.columns]
#         if missing_required:
#             print(f"❌ Missing required columns in {filepath}: {missing_required}")
#             return pd.DataFrame()
            
#         # Add missing optional columns with defaults
#         for col in optional_cols:
#             if col not in df.columns:
#                 if col == 'source_rule':
#                     df[col] = 'unknown'
#                 elif col in ['freq', 'edit_count', 'seed']:
#                     df[col] = 1
                    
#         return df
#     except Exception as e:
#         print(f"❌ Error loading {filepath}: {e}")
#         return pd.DataFrame()

# # Create sample data if files don't exist
# def create_sample_data():
#     """Create sample Spanish OCR normalization data for testing"""
#     print("Creating sample data for testing...")
    
#     # Sample Spanish OCR errors and corrections
#     sample_data = [
#         ("hola", "hola", "identity", 1000, 0, 1),
#         ("h0la", "hola", "digit_confusion", 800, 1, 1),
#         ("casa", "casa", "identity", 950, 0, 1),
#         ("c4sa", "casa", "digit_confusion", 750, 1, 1),
#         ("perr0", "perro", "digit_confusion", 600, 1, 1),
#         ("amig0", "amigo", "digit_confusion", 700, 1, 1),
#         ("niñ0", "niño", "digit_confusion", 500, 1, 1),
#         ("españ0l", "español", "digit_confusion", 400, 1, 1),
#         ("tiemp0", "tiempo", "digit_confusion", 650, 1, 1),
#         ("mañana", "mañana", "identity", 900, 1, 1),
#     ]
    
#     # Expand sample data significantly for training
#     expanded_data = []
#     for inp, tgt, rule, freq, edit, seed in sample_data:
#         # Add original
#         expanded_data.append((inp, tgt, rule, freq, edit, seed))
#         # Add many variations with different seeds
#         for i in range(2, 50):  # Create more variations
#             expanded_data.append((inp, tgt, rule, freq, edit, i))
    
#     df = pd.DataFrame(expanded_data, columns=['input', 'target', 'source_rule', 'freq', 'edit_count', 'seed'])
    
#     # Split into train/dev/test
#     total_size = len(df)
#     train_size = int(0.7 * total_size)
#     dev_size = int(0.15 * total_size)
    
#     train_df = df[:train_size].copy()
#     dev_df = df[train_size:train_size + dev_size].copy()
#     test_df = df[train_size + dev_size:].copy()
    
#     return train_df, dev_df, test_df

# # Try to load data files, create samples if not found
# try:
#     train_df = load_tsv_data('/kaggle/input/data-synthetic-noise/train.tsv')
#     dev_df = load_tsv_data('/kaggle/input/data-synthetic-noise/dev.tsv')
#     test_df = load_tsv_data('/kaggle/input/data-synthetic-noise/test.tsv')
    
#     if len(train_df) == 0:
#         print("⚠ No training data found, creating sample data...")
#         train_df, dev_df, test_df = create_sample_data()
        
# except FileNotFoundError:
#     print("⚠ Data files not found, creating sample data...")
#     train_df, dev_df, test_df = create_sample_data()

# # Save data for reference
# os.makedirs('/kaggle/working', exist_ok=True)
# train_df.to_csv('/kaggle/working/train.tsv', sep='\t', index=False, encoding='utf-8')
# dev_df.to_csv('/kaggle/working/dev.tsv', sep='\t', index=False, encoding='utf-8')
# test_df.to_csv('/kaggle/working/test.tsv', sep='\t', index=False, encoding='utf-8')

# print(f"Training samples: {len(train_df)}")
# print(f"Dev samples: {len(dev_df)}")
# print(f"Test samples: {len(test_df)}")

# if len(train_df) > 0:
#     print("\nSample training data:")
#     print(train_df[['input', 'target', 'source_rule']].head(3))
#     print(f"\nRule distribution (top 10):")
#     print(train_df['source_rule'].value_counts().head(10))

# def clean_text(text):
#     """Basic text cleaning: NFC normalization + basic filtering"""
#     if pd.isna(text):
#         return ""
#     # Unicode NFC normalization
#     text = unicodedata.normalize('NFC', str(text))
#     # Keep Spanish chars + basic punctuation
#     allowed_chars = set('abcdefghijklmnopqrstuvwxyzABCDEFGHIJKLMNOPQRSTUVWXYZ'
#                        'áéíóúüñÁÉÍÓÚÜÑàèìòùÀÈÌÒÙ'
#                        '0123456789 .,;:!?¿¡()[]{}"-\'')
#     text = ''.join(c for c in text if c in allowed_chars)
#     return text.strip()

# def clean_dataset(df, name):
#     """Clean and deduplicate dataset"""
#     if len(df) == 0:
#         return df
        
#     print(f"\nCleaning {name} dataset...")
#     original_size = len(df)
    
#     # Clean text columns
#     df['input'] = df['input'].apply(clean_text)
#     df['target'] = df['target'].apply(clean_text)
    
#     # Remove empty or invalid entries
#     df = df[(df['input'].str.len() > 0) & (df['target'].str.len() > 0)]
#     df = df[df['input'].str.len() <= 100]  # Max length filter
#     df = df[df['target'].str.len() <= 100]
    
#     # Deduplicate
#     df = df.drop_duplicates(subset=['input', 'target'])
    
#     print(f"  {original_size} → {len(df)} samples after cleaning")
    
#     if len(df) > 0:
#         # Edit distance stats
#         edit_distances = df.apply(lambda row: rapidfuzz.distance.Levenshtein.distance(
#             row['input'], row['target']), axis=1)
#         print(f"  Edit distance: mean={edit_distances.mean():.1f}, max={edit_distances.max()}")
        
#         # Rule distribution
#         if 'source_rule' in df.columns:
#             print(f"  Unique rules: {df['source_rule'].nunique()}")
    
#     return df.reset_index(drop=True)

# # Clean all datasets
# train_df = clean_dataset(train_df, "train")
# dev_df = clean_dataset(dev_df, "dev") 
# test_df = clean_dataset(test_df, "test")

# print("✓ Cleaned datasets")

# # Model selection - choose between byt5-small (character-level) or t5-small
# MODEL_NAME = "google/byt5-small"
# print(f"Using base model: {MODEL_NAME}")

# # Load tokenizer
# try:
#     tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
#     print(f"Tokenizer vocab size: {tokenizer.vocab_size}")
# except Exception as e:
#     print(f"❌ Error loading tokenizer: {e}")
#     exit(1)

# # Task prefix for T5
# PREFIX = "fix_es: "

# # Load or create lexicon and frequency table
# lexicon = set()
# freq_map = {}

# try:
#     with open('spanish_lexicon.txt', 'r', encoding='utf-8') as f:
#         lexicon = set(line.strip() for line in f if line.strip())
#     print(f"Loaded lexicon: {len(lexicon)} words")
# except FileNotFoundError:
#     print("⚠ spanish_lexicon.txt not found - creating lexicon from training data")
#     if len(train_df) > 0:
#         lexicon = set(train_df['target'].tolist())
#         # Save for future use
#         with open('spanish_lexicon.txt', 'w', encoding='utf-8') as f:
#             for word in sorted(lexicon):
#                 f.write(f"{word}\n")
#     print(f"Created lexicon: {len(lexicon)} words")

# try:
#     freq_df = pd.read_csv('freq_table.tsv', sep='\t')
#     freq_map = dict(zip(freq_df.iloc[:, 0], freq_df.iloc[:, 1]))
#     print(f"Loaded frequency table: {len(freq_map)} entries")
# except FileNotFoundError:
#     print("⚠ freq_table.tsv not found - using uniform frequencies")
#     freq_map = {word: 1.0 for word in lexicon}
#     # Create basic frequency table
#     if len(train_df) > 0:
#         freq_counts = train_df['target'].value_counts()
#         freq_map.update(freq_counts.to_dict())
#         # Save for future use
#         freq_df = pd.DataFrame(list(freq_map.items()), columns=['word', 'frequency'])
#         freq_df.to_csv('freq_table.tsv', sep='\t', index=False)

# # Convert lexicon to list for rapidfuzz
# lexicon_list = list(lexicon)
# print("✓ Lexicon and frequencies loaded")

# MAX_LENGTH = 32

# def preprocess_function(examples):
#     """Tokenize inputs and targets with T5 prefix"""
#     # Add prefix to inputs
#     inputs = [PREFIX + inp for inp in examples['input']]
#     targets = examples['target']
    
#     # Tokenize
#     model_inputs = tokenizer(inputs, max_length=MAX_LENGTH, truncation=True, padding=False)
#     labels = tokenizer(targets, max_length=MAX_LENGTH, truncation=True, padding=False)
    
#     # T5 expects labels, not target_ids
#     model_inputs['labels'] = labels['input_ids']
#     return model_inputs

# # Convert to HuggingFace datasets
# print("Converting to HF datasets and preprocessing...")

# if len(train_df) > 0:
#     train_dataset = Dataset.from_pandas(train_df[['input', 'target']])
#     train_dataset = train_dataset.map(preprocess_function, batched=True, remove_columns=['input', 'target'])
#     print(f"Train dataset: {len(train_dataset)} samples")

# if len(dev_df) > 0:
#     dev_dataset = Dataset.from_pandas(dev_df[['input', 'target']])  
#     dev_dataset = dev_dataset.map(preprocess_function, batched=True, remove_columns=['input', 'target'])
#     print(f"Dev dataset: {len(dev_dataset)} samples")

# if len(test_df) > 0:
#     test_dataset = Dataset.from_pandas(test_df[['input', 'target']])
#     test_dataset = test_dataset.map(preprocess_function, batched=True, remove_columns=['input', 'target'])
#     print(f"Test dataset: {len(test_dataset)} samples")

# print("✓ Datasets preprocessed and tokenized")

# print("Loading base model...")

# # Load model with proper settings
# model_kwargs = {
#     'torch_dtype': torch.float16 if torch.cuda.is_available() else torch.float32,
#     'device_map': None  # We'll handle device placement manually
# }

# try:
#     model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME, **model_kwargs)
#     print(f"✓ Base model loaded successfully")
# except Exception as e:
#     print(f"❌ Error loading model: {e}")
#     exit(1)

# # CRITICAL FIX: Configure LoRA BEFORE moving to CUDA
# print("Configuring LoRA...")
# lora_config = LoraConfig(
#     r=16,                              # Increased rank for better performance
#     lora_alpha=32,                     # Increased alpha
#     target_modules=["q", "k", "v", "o"],  # All attention layers
#     lora_dropout=0.05,                 # Reduced dropout for stability
#     bias="none",                       
#     task_type=TaskType.SEQ_2_SEQ_LM,   
# )

# # Apply LoRA to model BEFORE moving to CUDA
# model = get_peft_model(model, lora_config)
# print("✓ LoRA applied to model")

# # NOW move to CUDA
# if torch.cuda.is_available():
#     model = model.cuda()
#     print("✓ Model moved to CUDA")

# # CRITICAL FIX: Properly enable gradients for LoRA parameters
# model.train()  # Set to training mode
# for name, param in model.named_parameters():
#     if 'lora' in name.lower():
#         param.requires_grad = True
#         print(f"✓ Enabled gradients for: {name}")
#     else:
#         param.requires_grad = False

# # Verify gradient setup
# trainable_params = [p for p in model.parameters() if p.requires_grad]
# total_trainable = sum(p.numel() for p in trainable_params)
# total_params = sum(p.numel() for p in model.parameters())

# print(f"Trainable parameters: {len(trainable_params)} tensors")
# print(f"Trainable parameter count: {total_trainable:,}")
# print(f"Total parameters: {total_params:,}")
# print(f"Trainable percentage: {100 * total_trainable / total_params:.4f}%")

# if len(trainable_params) == 0:
#     print("❌ No trainable parameters found!")
#     exit(1)

# # Test gradient computation
# print("Testing gradient computation...")
# try:
#     # Create a small test batch
#     test_input_ids = torch.randint(0, tokenizer.vocab_size, (1, 10)).cuda()
#     test_labels = torch.randint(0, tokenizer.vocab_size, (1, 10)).cuda()
    
#     # Forward pass
#     outputs = model(input_ids=test_input_ids, labels=test_labels)
#     loss = outputs.loss
    
#     # Backward pass test
#     loss.backward()
    
#     # Check if gradients exist
#     grad_count = 0
#     for name, param in model.named_parameters():
#         if param.requires_grad and param.grad is not None:
#             grad_count += 1
    
#     print(f"✓ Gradient test passed! {grad_count} parameters have gradients")
    
#     # Clear test gradients
#     model.zero_grad()
    
# except Exception as e:
#     print(f"❌ Gradient test failed: {e}")
#     exit(1)

# # Determine batch size based on available memory
# if torch.cuda.is_available():
#     gpu_memory = torch.cuda.get_device_properties(0).total_memory / 1e9
#     if gpu_memory < 15:  
#         batch_size = 2
#         grad_accum = 8
#     else:  
#         batch_size = 4
#         grad_accum = 4
# else:
#     batch_size = 1
#     grad_accum = 16

# print(f"Batch size: {batch_size}, Gradient accumulation: {grad_accum}")
# print(f"Effective batch size: {batch_size * grad_accum}")

# # FIXED Training arguments
# training_args = Seq2SeqTrainingArguments(
#     output_dir="./results",
#     overwrite_output_dir=True,
#     evaluation_strategy="no",
#     save_strategy="no",  # Disable saving during training to avoid issues
#     logging_steps=100,
#     per_device_train_batch_size=batch_size,
#     per_device_eval_batch_size=batch_size,
#     gradient_accumulation_steps=grad_accum,
#     learning_rate=1e-4,  # Reduced learning rate for stability
#     num_train_epochs=5,  # Start with just 1 epoch
#     warmup_steps=50,
#     weight_decay=0.01,
#     fp16=torch.cuda.is_available(),
#     dataloader_pin_memory=False,
#     remove_unused_columns=True,
#     report_to=[],
#     disable_tqdm=False,
#     dataloader_num_workers=0,
#     gradient_checkpointing=False,  # Disable to avoid complications
#     prediction_loss_only=True,
#     ddp_find_unused_parameters=False,
#     max_grad_norm=1.0,  # Add gradient clipping
# )

# print("✓ Training arguments configured")

# # Data collator for seq2seq
# data_collator = DataCollatorForSeq2Seq(
#     tokenizer=tokenizer,
#     model=model,
#     padding=True,
#     max_length=MAX_LENGTH,
#     pad_to_multiple_of=8 if training_args.fp16 else None,
# )

# # FIXED Trainer initialization
# trainer = Seq2SeqTrainer(
#     model=model,
#     args=training_args,
#     train_dataset=train_dataset if len(train_df) > 0 else None,
#     eval_dataset=None,  # No evaluation during training
#     tokenizer=tokenizer,
#     data_collator=data_collator,
# )

# print("✓ Trainer initialized")

# # Training with better error handling
# if len(train_df) > 0:
#     print("Starting training...")
#     print(f"Training on {len(train_df)} samples for {training_args.num_train_epochs} epochs")
    
#     try:
#         # Clear GPU cache before training
#         if torch.cuda.is_available():
#             torch.cuda.empty_cache()
            
#         # Ensure model is in training mode
#         model.train()
        
#         # Run training
#         train_result = trainer.train()
#         print(f"✓ Training completed!")
#         print(f"Final train loss: {train_result.training_loss:.4f}")
        
#         # Save training metrics
#         os.makedirs("./results", exist_ok=True)
#         with open("./results/training_results.json", "w") as f:
#             json.dump(train_result.metrics, f, indent=2)
            
#         print("✓ Training results saved")
            
#     except Exception as e:
#         print(f"❌ Training failed: {e}")
#         import traceback
#         traceback.print_exc()
# else:
#     print("⚠ No training data available - skipping training")

# print("✓ Training phase complete")

# # Simple evaluation function without trainer.predict
# # def simple_evaluate(model, tokenizer, df, name, max_samples=100):
# #     """Simple evaluation without using trainer.predict"""
# #     if len(df) == 0:
# #         print(f"⚠ No {name} data to evaluate")
# #         return {}
    
# #     print(f"\nEvaluating on {name} set (max {max_samples} samples)...")
    
# #     # Take a subset for evaluation
# #     eval_df = df.head(max_samples) if len(df) > max_samples else df
    
# #     model.eval()  # Set to evaluation mode
# #     correct_count = 0
# #     total_char_errors = 0
# #     total_chars = 0
    
# #     with torch.no_grad():
# #         for idx, row in eval_df.iterrows():
# #             try:
# #                 # Prepare input
# #                 input_text = PREFIX + row['input']
# #                 target_text = row['target']
                
# #                 # Tokenize input
# #                 input_ids = tokenizer.encode(input_text, return_tensors="pt")
# #                 if torch.cuda.is_available():
# #                     input_ids = input_ids.cuda()
                
# #                 # Generate prediction
# #                 outputs = model.generate(
# #                     input_ids,
# #                     max_length=MAX_LENGTH,
# #                     num_beams=2,
# #                     early_stopping=True,
# #                     pad_token_id=tokenizer.pad_token_id,
# #                     eos_token_id=tokenizer.eos_token_id
# #                 )
                
# #                 # Decode prediction
# #                 pred_text = tokenizer.decode(outputs[0], skip_special_tokens=True).strip()
                
# #                 # Calculate metrics
# #                 if pred_text == target_text:
# #                     correct_count += 1
                
# #                 char_errors = rapidfuzz.distance.Levenshtein.distance(pred_text, target_text)
# #                 total_char_errors += char_errors
# #                 total_chars += len(target_text)
                
# #                 # Print first few examples
# #                 if idx < 3:
# #                     status = "✓" if pred_text == target_text else "✗"
# #                     print(f"  {row['input']} → {target_text} | Pred: {pred_text} {status}")
                    
# #             except Exception as e:
# #                 print(f"  Warning: Error processing sample {idx}: {e}")
# #                 continue
    
# #     # Calculate final metrics
# #     exact_match = correct_count / len(eval_df) if len(eval_df) > 0 else 0
# #     cer = total_char_errors / max(total_chars, 1)
    
# #     metrics = {
# #         'exact_match': exact_match,
# #         'character_error_rate': cer,
# #         'samples_evaluated': len(eval_df)
# #     }
    
# #     print(f"  Exact Match: {exact_match:.4f}")
# #     print(f"  CER: {cer:.4f}")
# #     print(f"  Samples: {len(eval_df)}")
    
# #     return metrics

# # # Evaluate on test set
# # if len(test_df) > 0:
# #     test_metrics = simple_evaluate(model, tokenizer, test_df, "test", max_samples=50)
# # else:
# #     test_metrics = {}

# # # Save evaluation report
# # eval_report = {
# #     'test_metrics': test_metrics,
# #     'model_name': MODEL_NAME,
# #     'lora_config': {
# #         'r': lora_config.r,
# #         'lora_alpha': lora_config.lora_alpha,
# #         'target_modules': lora_config.target_modules,
# #         'lora_dropout': lora_config.lora_dropout,
# #         'bias': lora_config.bias,
# #     }
# # }

# # try:
# #     with open("eval_report.json", "w") as f:
# #         json.dump(eval_report, f, indent=2)
# #     print("✓ Evaluation report saved")
# # except Exception as e:
# #     print(f"Warning: Could not save eval report: {e}")

# # print("✓ Evaluation complete")

# # def correct_word(word, num_beams=4):
# #     """
# #     Normalize a single word using the trained model
# #     Returns: (prediction, confidence)
# #     """
# #     # Clean input
# #     clean_word = clean_text(word)
# #     if not clean_word:
# #         return word, 0.0
    
# #     try:
# #         model.eval()  # Set to evaluation mode
        
# #         # Tokenize
# #         input_text = PREFIX + clean_word
# #         input_ids = tokenizer.encode(input_text, return_tensors="pt")
# #         if torch.cuda.is_available():
# #             input_ids = input_ids.cuda()
        
# #         # Generate
# #         with torch.no_grad():
# #             outputs = model.generate(
# #                 input_ids,
# #                 max_length=MAX_LENGTH,
# #                 num_beams=num_beams,
# #                 early_stopping=True,
# #                 no_repeat_ngram_size=2,
# #                 do_sample=False,
# #                 pad_token_id=tokenizer.pad_token_id,
# #                 eos_token_id=tokenizer.eos_token_id
# #             )
            
# #             prediction = tokenizer.decode(outputs[0], skip_special_tokens=True).strip()
# #             confidence = 1.0  # Placeholder - could compute actual confidence
        
# #         return prediction, confidence
        
# #     except Exception as e:
# #         print(f"❌ Error in correction: {e}")
# #         return word, 0.0

# # print("✓ Inference utility ready")

# # # Save model and adapter
# # print("Saving model and adapter...")

# # os.makedirs("./model_out", exist_ok=True)

# # try:
# #     # Save the LoRA adapter
# #     model.save_pretrained("./model_out/lora_adapter")
# #     tokenizer.save_pretrained("./model_out/tokenizer")
    
# #     # Save model configuration
# #     config_info = {
# #         "base_model": MODEL_NAME,
# #         "task_prefix": PREFIX,
# #         "max_length": MAX_LENGTH,
# #         "lora_config": {
# #             'r': lora_config.r,
# #             'lora_alpha': lora_config.lora_alpha,
# #             'target_modules': lora_config.target_modules,
# #             'lora_dropout': lora_config.lora_dropout,
# #             'bias': lora_config.bias,
# #         },
# #         "training_args": {
# #             "learning_rate": training_args.learning_rate,
# #             "num_epochs": training_args.num_train_epochs,
# #             "batch_size": training_args.per_device_train_batch_size
# #         }
# #     }
    
# #     with open("./model_out/model_config.json", "w") as f:
# #         json.dump(config_info, f, indent=2)
    
# #     print("✓ LoRA adapter and config saved to ./model_out/")
    
# # except Exception as e:
# #     print(f"❌ Error saving model: {e}")

# # # Smoke test
# # if len(test_df) > 0:
# #     print("🔥 SMOKE TEST - Sample Corrections:")
# #     print("=" * 60)
    
# #     # Take sample of test data for smoke test
# #     sample_size = min(10, len(test_df))
# #     sample_df = test_df.sample(n=sample_size, random_state=42).reset_index(drop=True)
    
# #     correct_count = 0
    
# #     for idx, row in sample_df.iterrows():
# #         input_word = row['input']
# #         gold_word = row['target']
        
# #         # Get model prediction
# #         pred, confidence = correct_word(input_word)
        
# #         is_correct = pred == gold_word
# #         if is_correct:
# #             correct_count += 1
        
# #         status = "✓" if is_correct else "✗"
# #         print(f"{idx+1:2d}. {input_word:15} → {gold_word:15} | Pred: {pred:15} {status}")
    
# #     accuracy = correct_count / len(sample_df)
# #     print("=" * 60)
# #     print(f"📊 SMOKE TEST ACCURACY: {accuracy:.1%} ({correct_count}/{len(sample_df)})")
    
# # else:
# #     print("⚠ No test data available for smoke test")

# # print("\n" + "="*60)
# # print("🎉 SPANISH OCR NORMALIZATION TRAINING COMPLETE!")  
# # print("="*60)

# # if len(train_df) > 0:
# #     print(f"✅ Trained on {len(train_df)} samples")
# #     print(f"✅ LoRA adapter saved to ./model_out/lora_adapter")
# #     print(f"✅ Tokenizer saved to ./model_out/tokenizer") 
# #     print(f"✅ Config saved to ./model_out/model_config.json")
    
# #     if test_metrics:
# #         test_em = test_metrics.get('exact_match', 0)  
# #         print(f"📊 Test Exact Match: {test_em:.1%}")
        
# #     trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
# #     print(f"💾 Model size: ~{trainable_params / 1e6:.1f}M trainable params")
    
# # else:
# #     print("⚠ No training performed - check data files")

# # print("\n📁 FILES CREATED:")
# # print("   - ./model_out/lora_adapter/ (LoRA weights)")  
# # print("   - ./model_out/tokenizer/ (tokenizer)")
# # print("   - ./model_out/model_config.json (config)")
# # print("   - eval_report.json (metrics)")

# # print("\n🚀 READY FOR USE!")
# # print("   Use correct_word(text) function for inference")
# # print("   Or load saved model for production use")

✓ Environment setup complete
Python version: 3.11.13 (main, Jun  4 2025, 08:57:29) [GCC 11.4.0]
PyTorch version: 2.6.0+cu124
CUDA available: True
GPU: Tesla P100-PCIE-16GB
GPU Memory: 17.1 GB
Current GPU memory: 0.0 MB
Quantization available: True
Training samples: 109112
Dev samples: 13639
Test samples: 13639

Sample training data:
         input       target    source_rule
0     intranel     intranet  ocr_confusion
1  evrasiático  eurasiático     historical
2    coñsumido    consumido  ocr_confusion

Rule distribution (top 10):
source_rule
ocr_confusion    45009
typo             34098
historical       16366
identity         13639
Name: count, dtype: int64

Cleaning train dataset...
  109112 → 81376 samples after cleaning
  Edit distance: mean=1.1, max=5
  Unique rules: 4

Cleaning dev dataset...
  13639 → 13021 samples after cleaning
  Edit distance: mean=1.1, max=5
  Unique rules: 4

Cleaning test dataset...
  13639 → 12992 samples after cleaning
  Edit distance: mean=1.1, max=5
  U

Tokenizer vocab size: 256
⚠ spanish_lexicon.txt not found - creating lexicon from training data
Created lexicon: 62018 words
⚠ freq_table.tsv not found - using uniform frequencies
✓ Lexicon and frequencies loaded
Converting to HF datasets and preprocessing...


Map:   0%|          | 0/81376 [00:00<?, ? examples/s]

Train dataset: 81376 samples


Map:   0%|          | 0/13021 [00:00<?, ? examples/s]

...

Map:   0%|          | 0/12992 [00:00<?, ? examples/s]

Test dataset: 12992 samples
✓ Datasets preprocessed and tokenized
Loading base model...


✓ Base model loaded successfully
Configuring LoRA...
✓ LoRA applied to model
✓ Model moved to CUDA
✓ Enabled gradients for: base_model.model.encoder.block.0.layer.0.SelfAttention.q.lora_A.default.weight
✓ Enabled gradients for: base_model.model.encoder.block.0.layer.0.SelfAttention.q.lora_B.default.weight
✓ Enabled gradients for: base_model.model.encoder.block.0.layer.0.SelfAttention.k.lora_A.default.weight
✓ Enabled gradients for: base_model.model.encoder.block.0.layer.0.SelfAttention.k.lora_B.default.weight
✓ Enabled gradients for: base_model.model.encoder.block.0.layer.0.SelfAttention.v.lora_A.default.weight
✓ Enabled gradients for: base_model.model.encoder.block.0.layer.0.SelfAttention.v.lora_B.default.weight
✓ Enabled gradients for: base_model.model.encoder.block.0.layer.0.SelfAttention.o.lora_A.default.weight
✓ Enabled gradients for: base_model.model.encoder.block.0.layer.0.SelfAttention.o.lora_B.default.weight
✓ Enabled gradients for: base_model.model.encoder.block.1.layer.0.Sel

Step,Training Loss
100,176.324600
200,209.420400
300,0.000000
400,0.000000
500,0.000000
600,0.000000
700,0.000000
800,0.000000


KeyboardInterrupt: 

In [11]:
# Simple evaluation function without trainer.predict
def simple_evaluate(model, tokenizer, df, name, max_samples=100):
    """Simple evaluation without using trainer.predict"""
    if len(df) == 0:
        print(f"⚠ No {name} data to evaluate")
        return {}
    
    print(f"\nEvaluating on {name} set (max {max_samples} samples)...")
    
    # Take a subset for evaluation
    eval_df = df.head(max_samples) if len(df) > max_samples else df
    
    model.eval()  # Set to evaluation mode
    correct_count = 0
    total_char_errors = 0
    total_chars = 0
    
    with torch.no_grad():
        for idx, row in eval_df.iterrows():
            try:
                # Prepare input
                input_text = PREFIX + row['input']
                target_text = row['target']
                
                # Tokenize input
                input_ids = tokenizer.encode(input_text, return_tensors="pt")
                if torch.cuda.is_available():
                    input_ids = input_ids.cuda()
                
                # FIXED: Generate prediction with keyword arguments
                outputs = model.generate(
                    input_ids=input_ids,  # Use keyword argument
                    max_length=MAX_LENGTH,
                    num_beams=2,
                    early_stopping=True,
                    pad_token_id=tokenizer.pad_token_id,
                    eos_token_id=tokenizer.eos_token_id
                )
                
                # Decode prediction
                pred_text = tokenizer.decode(outputs[0], skip_special_tokens=True).strip()
                
                # Calculate metrics
                if pred_text == target_text:
                    correct_count += 1
                
                char_errors = rapidfuzz.distance.Levenshtein.distance(pred_text, target_text)
                total_char_errors += char_errors
                total_chars += len(target_text)
                
                # Print first few examples
                if idx < 3:
                    status = "✓" if pred_text == target_text else "✗"
                    print(f"  {row['input']} → {target_text} | Pred: {pred_text} {status}")
                    
            except Exception as e:
                print(f"  Warning: Error processing sample {idx}: {e}")
                continue
    
    # Calculate final metrics
    exact_match = correct_count / len(eval_df) if len(eval_df) > 0 else 0
    cer = total_char_errors / max(total_chars, 1)
    
    metrics = {
        'exact_match': exact_match,
        'character_error_rate': cer,
        'samples_evaluated': len(eval_df)
    }
    
    print(f"  Exact Match: {exact_match:.4f}")
    print(f"  CER: {cer:.4f}")
    print(f"  Samples: {len(eval_df)}")
    
    return metrics

# Evaluate on test set
if len(test_df) > 0:
    test_metrics = simple_evaluate(model, tokenizer, test_df, "test", max_samples=50)
else:
    test_metrics = {}

# Save evaluation report
eval_report = {
    'test_metrics': test_metrics,
    'model_name': MODEL_NAME,
    'lora_config': {
        'r': lora_config.r,
        'lora_alpha': lora_config.lora_alpha,
        'target_modules': list(lora_config.target_modules),  # Convert set to list
        'lora_dropout': lora_config.lora_dropout,
        'bias': lora_config.bias,
    }
}

try:
    with open("eval_report.json", "w") as f:
        json.dump(eval_report, f, indent=2)
    print("✓ Evaluation report saved")
except Exception as e:
    print(f"Warning: Could not save eval report: {e}")

print("✓ Evaluation complete")

def correct_word(word, num_beams=4):
    """
    Normalize a single word using the trained model
    Returns: (prediction, confidence)
    """
    # Clean input
    clean_word = clean_text(word)
    if not clean_word:
        return word, 0.0
    
    try:
        model.eval()  # Set to evaluation mode
        
        # Tokenize
        input_text = PREFIX + clean_word
        input_ids = tokenizer.encode(input_text, return_tensors="pt")
        if torch.cuda.is_available():
            input_ids = input_ids.cuda()
        
        # FIXED: Generate with keyword arguments
        with torch.no_grad():
            outputs = model.generate(
                input_ids=input_ids,  # Use keyword argument
                max_length=MAX_LENGTH,
                num_beams=num_beams,
                early_stopping=True,
                no_repeat_ngram_size=2,
                do_sample=False,
                pad_token_id=tokenizer.pad_token_id,
                eos_token_id=tokenizer.eos_token_id
            )
            
            prediction = tokenizer.decode(outputs[0], skip_special_tokens=True).strip()
            confidence = 1.0  # Placeholder - could compute actual confidence
        
        return prediction, confidence
        
    except Exception as e:
        print(f"❌ Error in correction: {e}")
        return word, 0.0

print("✓ Inference utility ready")

# Save model and adapter
print("Saving model and adapter...")

os.makedirs("./model_out", exist_ok=True)

try:
    # Save the LoRA adapter
    model.save_pretrained("./model_out/lora_adapter")
    tokenizer.save_pretrained("./model_out/tokenizer")
    
    # Save model configuration
    config_info = {
        "base_model": MODEL_NAME,
        "task_prefix": PREFIX,
        "max_length": MAX_LENGTH,
        "lora_config": {
            'r': lora_config.r,
            'lora_alpha': lora_config.lora_alpha,
            'target_modules': lora_config.target_modules,
            'lora_dropout': lora_config.lora_dropout,
            'bias': lora_config.bias,
        },
        "training_args": {
            "learning_rate": training_args.learning_rate,
            "num_epochs": training_args.num_train_epochs,
            "batch_size": training_args.per_device_train_batch_size
        }
    }
    
    with open("./model_out/model_config.json", "w") as f:
        json.dump(config_info, f, indent=2)
    
    print("✓ LoRA adapter and config saved to ./model_out/")
    
except Exception as e:
    print(f"❌ Error saving model: {e}")

# Smoke test
if len(test_df) > 0:
    print("🔥 SMOKE TEST - Sample Corrections:")
    print("=" * 60)
    
    # Take sample of test data for smoke test
    sample_size = min(10, len(test_df))
    sample_df = test_df.sample(n=sample_size, random_state=42).reset_index(drop=True)
    
    correct_count = 0
    
    for idx, row in sample_df.iterrows():
        input_word = row['input']
        gold_word = row['target']
        
        # Get model prediction
        pred, confidence = correct_word(input_word)
        
        is_correct = pred == gold_word
        if is_correct:
            correct_count += 1
        
        status = "✓" if is_correct else "✗"
        print(f"{idx+1:2d}. {input_word:15} → {gold_word:15} | Pred: {pred:15} {status}")
    
    accuracy = correct_count / len(sample_df)
    print("=" * 60)
    print(f"📊 SMOKE TEST ACCURACY: {accuracy:.1%} ({correct_count}/{len(sample_df)})")
    
else:
    print("⚠ No test data available for smoke test")

print("\n" + "="*60)
print("🎉 SPANISH OCR NORMALIZATION TRAINING COMPLETE!")  
print("="*60)

if len(train_df) > 0:
    print(f"✅ Trained on {len(train_df)} samples")
    print(f"✅ LoRA adapter saved to ./model_out/lora_adapter")
    print(f"✅ Tokenizer saved to ./model_out/tokenizer") 
    print(f"✅ Config saved to ./model_out/model_config.json")
    
    if test_metrics:
        test_em = test_metrics.get('exact_match', 0)  
        print(f"📊 Test Exact Match: {test_em:.1%}")
        
    trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"💾 Model size: ~{trainable_params / 1e6:.1f}M trainable params")
    
else:
    print("⚠ No training performed - check data files")

print("\n📁 FILES CREATED:")
print("   - ./model_out/lora_adapter/ (LoRA weights)")  
print("   - ./model_out/tokenizer/ (tokenizer)")
print("   - ./model_out/model_config.json (config)")
print("   - eval_report.json (metrics)")

print("\n🚀 READY FOR USE!")
print("   Use correct_word(text) function for inference")
print("   Or load saved model for production use")


Evaluating on test set (max 50 samples)...
  oerebróvafcular → cerebrovascular | Pred: oerebróvafcular ✗
  mámuts → mamuts | Pred: muts ✗
  agropecurios → agropecuarios | Pred: agropecurios ✗
  Exact Match: 0.0400
  CER: 0.2934
  Samples: 50
✓ Evaluation report saved
✓ Evaluation complete
✓ Inference utility ready
Saving model and adapter...
❌ Error saving model: Object of type set is not JSON serializable
🔥 SMOKE TEST - Sample Corrections:
 1. méco            → meco            | Pred: méco            ✗
 2. ravenj          → raven           | Pred: renj            ✗
 3. peuqeñín        → pequeñín        | Pred: peuqeñin        ✗
 4. íugares         → lugares         | Pred: uugares         ✗
 5. bivirla         → vivirla         | Pred: bivirla         ✗
 6. autlrregular    → autorregular    | Pred: autlrregular    ✗
 7. ecotógicas      → ecológicas      | Pred: otógicas        ✗
 8. añacrónico      → anacrónico      | Pred: acrónico        ✗
 9. deíicado        → delicado        | Pr

In [11]:
import os
import json
import random
import numpy as np
import pandas as pd
from tqdm.auto import tqdm
import torch
from transformers import (
    AutoTokenizer, AutoModelForSeq2SeqLM, Seq2SeqTrainer, Seq2SeqTrainingArguments,
    DataCollatorForSeq2Seq, TrainerCallback
)
from datasets import Dataset
from peft import LoraConfig, get_peft_model, TaskType, PeftModel
import rapidfuzz.distance
import unicodedata
import warnings
warnings.filterwarnings("ignore")

def seed_everything(seed=42):
    """Set seeds for reproducibility"""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)

seed_everything(42)
print("Environment setup complete")

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

def load_tsv_data(filepath):
    """Load TSV data"""
    try:
        df = pd.read_csv(filepath, sep='\t', encoding='utf-8')
        required_cols = ['input', 'target']
        
        missing_required = [col for col in required_cols if col not in df.columns]
        if missing_required:
            print(f"Missing required columns in {filepath}: {missing_required}")
            return pd.DataFrame()
        
        return df
    except Exception as e:
        print(f"Error loading {filepath}: {e}")
        return pd.DataFrame()

# Load data
try:
    train_df = load_tsv_data('/kaggle/input/data-synthetic-noise/train.tsv')
    test_df = load_tsv_data('/kaggle/input/data-synthetic-noise/test.tsv')
    
    if len(train_df) == 0:
        print("No training data found")
        exit(1)
        
except FileNotFoundError:
    print("Data files not found")
    exit(1)

print(f"Training samples: {len(train_df)}")
print(f"Test samples: {len(test_df)}")

def clean_text(text):
    """Clean text"""
    if pd.isna(text):
        return ""
    text = unicodedata.normalize('NFC', str(text))
    allowed_chars = set('abcdefghijklmnopqrstuvwxyzABCDEFGHIJKLMNOPQRSTUVWXYZ'
                       'áéíóúüñÁÉÍÓÚÜÑàèìòùÀÈÌÒÙ'
                       '0123456789 .,;:!?¿¡()[]{}"-\'')
    text = ''.join(c for c in text if c in allowed_chars)
    return text.strip()

def prepare_dataset(df, name, max_samples=5000):
    """Prepare dataset with better filtering"""
    print(f"\nPreparing {name} dataset...")
    
    # Clean text
    df['input'] = df['input'].apply(clean_text)
    df['target'] = df['target'].apply(clean_text)
    
    # Remove empty entries
    df = df[(df['input'].str.len() > 0) & (df['target'].str.len() > 0)]
    df = df[df['input'].str.len() <= 50]  # Shorter max length
    df = df[df['target'].str.len() <= 50]
    
    # CRITICAL FIX: Remove most identity pairs that cause training collapse
    identity_mask = df['input'] == df['target']
    identity_count = identity_mask.sum()
    non_identity_count = (~identity_mask).sum()
    
    print(f"  Identity pairs: {identity_count}")
    print(f"  Non-identity pairs: {non_identity_count}")
    
    # Keep only 10% of identity pairs to prevent collapse
    if identity_count > 100:
        identity_samples = df[identity_mask].sample(n=min(100, identity_count), random_state=42)
        non_identity_samples = df[~identity_mask]
        df = pd.concat([non_identity_samples, identity_samples], ignore_index=True)
        print(f"  Reduced identity pairs to {len(identity_samples)}")
    
    # Sample if too large
    if len(df) > max_samples:
        df = df.sample(n=max_samples, random_state=42)
        print(f"  Sampled {max_samples} examples")
    
    print(f"  Final size: {len(df)}")
    
    # Show edit distance distribution
    if len(df) > 0:
        edit_distances = df.apply(lambda row: rapidfuzz.distance.Levenshtein.distance(
            row['input'], row['target']), axis=1)
        print(f"  Edit distance - mean: {edit_distances.mean():.2f}, max: {edit_distances.max()}")
        
        # Show sample data
        print(f"  Sample pairs:")
        for i in range(min(3, len(df))):
            print(f"    '{df.iloc[i]['input']}' -> '{df.iloc[i]['target']}'")
    
    return df.reset_index(drop=True)

# Prepare datasets
train_df = prepare_dataset(train_df, "train", max_samples=5000)
test_df = prepare_dataset(test_df, "test", max_samples=1000)

# Model setup
MODEL_NAME = "google/byt5-small"
print(f"\nUsing model: {MODEL_NAME}")

try:
    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
    print(f"Tokenizer loaded, vocab size: {tokenizer.vocab_size}")
except Exception as e:
    print(f"Error loading tokenizer: {e}")
    exit(1)

# CRITICAL: Use simpler prefix and validate tokenization
PREFIX = "fix: "
MAX_LENGTH = 64

print(f"Testing tokenization with prefix '{PREFIX}':")
test_input = PREFIX + "h0la"
test_target = "hola"
input_tokens = tokenizer(test_input, max_length=MAX_LENGTH, truncation=True)
target_tokens = tokenizer(test_target, max_length=MAX_LENGTH, truncation=True)
print(f"  Input: '{test_input}' -> {len(input_tokens['input_ids'])} tokens")
print(f"  Target: '{test_target}' -> {len(target_tokens['input_ids'])} tokens")

def preprocess_function(examples):
    """Tokenize with validation"""
    inputs = [PREFIX + inp for inp in examples['input']]
    targets = examples['target']
    
    model_inputs = tokenizer(inputs, max_length=MAX_LENGTH, truncation=True, padding=False)
    labels = tokenizer(targets, max_length=MAX_LENGTH, truncation=True, padding=False)
    
    # Validation
    valid_indices = []
    for i, (inp_ids, lbl_ids) in enumerate(zip(model_inputs['input_ids'], labels['input_ids'])):
        if len(inp_ids) > 0 and len(lbl_ids) > 0:
            valid_indices.append(i)
    
    if len(valid_indices) != len(model_inputs['input_ids']):
        print(f"Filtered out {len(model_inputs['input_ids']) - len(valid_indices)} invalid samples")
    
    # Keep only valid samples
    model_inputs['input_ids'] = [model_inputs['input_ids'][i] for i in valid_indices]
    model_inputs['attention_mask'] = [model_inputs['attention_mask'][i] for i in valid_indices]
    model_inputs['labels'] = [labels['input_ids'][i] for i in valid_indices]
    
    return model_inputs

# Create datasets
print("\nCreating datasets...")
train_dataset = Dataset.from_pandas(train_df[['input', 'target']])
train_dataset = train_dataset.map(preprocess_function, batched=True, remove_columns=['input', 'target'])
print(f"Train dataset: {len(train_dataset)} samples")

test_dataset = Dataset.from_pandas(test_df[['input', 'target']])
test_dataset = test_dataset.map(preprocess_function, batched=True, remove_columns=['input', 'target'])
print(f"Test dataset: {len(test_dataset)} samples")

# Show sample tokenized data
print("\nSample tokenized data:")
for i in range(min(2, len(train_dataset))):
    sample = train_dataset[i]
    input_text = tokenizer.decode(sample['input_ids'], skip_special_tokens=True)
    label_text = tokenizer.decode(sample['labels'], skip_special_tokens=True)
    print(f"  Input: '{input_text}'")
    print(f"  Label: '{label_text}'")

# Load model
print("\nLoading model...")
try:
    model = AutoModelForSeq2SeqLM.from_pretrained(
        MODEL_NAME,
        torch_dtype=torch.float32,  # Use float32 for stability
    )
    print("Base model loaded")
except Exception as e:
    print(f"Error loading model: {e}")
    exit(1)

# VERY CONSERVATIVE LoRA config
lora_config = LoraConfig(
    r=4,                               # Very small rank
    lora_alpha=4,                      # Small alpha
    target_modules=["q", "v"],         # Only 2 modules
    lora_dropout=0.05,                 
    bias="none",                       
    task_type=TaskType.SEQ_2_SEQ_LM,   
)

print("Applying LoRA...")
model = get_peft_model(model, lora_config)

if torch.cuda.is_available():
    model = model.cuda()
    print("Model moved to CUDA")

# Enable training mode
model.train()
for name, param in model.named_parameters():
    if 'lora' in name.lower():
        param.requires_grad = True

trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
total_params = sum(p.numel() for p in model.parameters())
print(f"Trainable: {trainable_params:,} ({100*trainable_params/total_params:.3f}%)")

# Test forward pass
print("\nTesting model...")
try:
    sample = train_dataset[0]
    input_ids = torch.tensor([sample['input_ids']]).cuda() if torch.cuda.is_available() else torch.tensor([sample['input_ids']])
    labels = torch.tensor([sample['labels']]).cuda() if torch.cuda.is_available() else torch.tensor([sample['labels']])
    
    with torch.no_grad():
        outputs = model(input_ids=input_ids, labels=labels)
        print(f"Forward pass successful, loss: {outputs.loss:.4f}")
        
except Exception as e:
    print(f"Model test failed: {e}")
    exit(1)

# VERY CONSERVATIVE training settings
training_args = Seq2SeqTrainingArguments(
    output_dir="./results",
    overwrite_output_dir=True,
    evaluation_strategy="no",
    save_strategy="no",
    logging_steps=100,
    per_device_train_batch_size=1,      # Smallest possible batch
    gradient_accumulation_steps=16,     # Large accumulation
    learning_rate=1e-4,                 # Very small learning rate
    num_train_epochs=5,                 # Just 1 epoch
    warmup_steps=50,
    weight_decay=0.01,
    fp16=False,                         # No mixed precision
    dataloader_pin_memory=False,
    remove_unused_columns=True,
    report_to=[],
    disable_tqdm=False,
    dataloader_num_workers=0,
    max_grad_norm=0.1,                  # Very strong gradient clipping
    prediction_loss_only=True,
)

print(f"Training settings: batch_size=1, grad_accum=16, lr=1e-5")

# Data collator
data_collator = DataCollatorForSeq2Seq(
    tokenizer=tokenizer,
    model=model,
    padding=True,
    max_length=MAX_LENGTH,
)

# Loss monitoring callback
class LossMonitorCallback(TrainerCallback):
    def __init__(self):
        self.losses = []
        
    def on_log(self, args, state, control, model=None, logs=None, **kwargs):
        if logs and 'train_loss' in logs:
            loss = logs['train_loss']
            step = state.global_step
            self.losses.append((step, loss))
            
            print(f"Step {step}: loss = {loss:.6f}")
            
            # Stop if loss becomes zero or explodes
            if loss == 0.0:
                print("ZERO LOSS - stopping training!")
                control.should_training_stop = True
            elif loss > 100:
                print("EXPLODING LOSS - stopping training!")
                control.should_training_stop = True

# Create trainer
trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    tokenizer=tokenizer,
    data_collator=data_collator,
)

# Add callback
loss_monitor = LossMonitorCallback()
trainer.add_callback(loss_monitor)

print("\nStarting training...")
try:
    train_result = trainer.train()
    print(f"Training completed! Final loss: {train_result.training_loss:.6f}")
    
    # Show loss progression
    if loss_monitor.losses:
        print("\nLoss progression:")
        for step, loss in loss_monitor.losses:
            print(f"  Step {step}: {loss:.6f}")
            
except Exception as e:
    print(f"Training failed: {e}")
    import traceback
    traceback.print_exc()

print("\nTesting inference...")

def test_model(input_text):
    """Test model inference"""
    try:
        model.eval()
        
        full_input = PREFIX + input_text
        input_ids = tokenizer.encode(full_input, return_tensors="pt")
        if torch.cuda.is_available():
            input_ids = input_ids.cuda()
        
        with torch.no_grad():
            outputs = model.generate(
                input_ids=input_ids,
                max_length=MAX_LENGTH,
                num_beams=1,
                do_sample=False,
                pad_token_id=tokenizer.pad_token_id,
                eos_token_id=tokenizer.eos_token_id
            )
            
            prediction = tokenizer.decode(outputs[0], skip_special_tokens=True).strip()
            return prediction
            
    except Exception as e:
        return f"ERROR: {e}"

# Test cases
test_cases = ["hola", "h0la", "casa", "c4sa"]
print("\nInference test:")
for test_input in test_cases:
    prediction = test_model(test_input)
    print(f"  '{test_input}' -> '{prediction}'")

# Test on actual data samples
print("\nTesting on real data samples:")
for i in range(min(3, len(test_df))):
    input_word = test_df.iloc[i]['input']
    target_word = test_df.iloc[i]['target']
    prediction = test_model(input_word)
    
    correct = "✓" if prediction == target_word else "✗"
    print(f"  '{input_word}' -> '{target_word}' | Pred: '{prediction}' {correct}")

print("\nDiagnostic complete!")
print("If you still see repetitive/garbled output, the fundamental training setup needs revision.")

Environment setup complete
PyTorch version: 2.6.0+cu124
CUDA available: True
GPU: Tesla P100-PCIE-16GB
Training samples: 109112
Test samples: 13639

Preparing train dataset...
  Identity pairs: 13639
  Non-identity pairs: 95471
  Reduced identity pairs to 100
  Sampled 5000 examples
  Final size: 5000
  Edit distance - mean: 1.22, max: 5
  Sample pairs:
    'escupeen' -> 'escupen'
    'cardnoso' -> 'carnoso'
    'pertiostio' -> 'periostio'

Preparing test dataset...
  Identity pairs: 1705
  Non-identity pairs: 11934
  Reduced identity pairs to 100
  Sampled 1000 examples
  Final size: 1000
  Edit distance - mean: 1.23, max: 4
  Sample pairs:
    'oerebróvafcular' -> 'cerebrovascular'
    'mámuts' -> 'mamuts'
    'agropecurios' -> 'agropecuarios'

Using model: google/byt5-small


Tokenizer loaded, vocab size: 256
Testing tokenization with prefix 'fix: ':
  Input: 'fix: h0la' -> 10 tokens
  Target: 'hola' -> 5 tokens

Creating datasets...


Map:   0%|          | 0/5000 [00:00<?, ? examples/s]

Train dataset: 5000 samples


Map:   0%|          | 0/1000 [00:00<?, ? examples/s]

Test dataset: 1000 samples

Sample tokenized data:
  Input: 'fix: escupeen'
  Label: 'escupen'
  Input: 'fix: cardnoso'
  Label: 'carnoso'

Loading model...


Base model loaded
Applying LoRA...
Model moved to CUDA
Trainable: 296,960 (0.099%)

Testing model...
Forward pass successful, loss: 5.3972
Training settings: batch_size=1, grad_accum=16, lr=1e-5

Starting training...


Step,Training Loss
100,6.549900
200,5.415600
300,4.185500
400,3.409000
500,2.954600
600,2.622600
700,2.306300
800,2.047500
900,1.819100
1000,1.710700


Step 1560: loss = 2.661345
Training completed! Final loss: 2.661345

Loss progression:
  Step 1560: 2.661345

Testing inference...

Inference test:
  'hola' -> 'ola'
  'h0la' -> 'h0la'
  'casa' -> 'casa'
  'c4sa' -> 'c4sa'

Testing on real data samples:
  'oerebróvafcular' -> 'cerebrovascular' | Pred: 'oerebróvafcular' ✗
  'mámuts' -> 'mamuts' | Pred: 'mamuts' ✓
  'agropecurios' -> 'agropecuarios' | Pred: 'agropecurios' ✗

Diagnostic complete!
If you still see repetitive/garbled output, the fundamental training setup needs revision.


In [4]:
!python -m bitsandbytes

False

===================================BUG REPORT===================================
/usr/local/lib/python3.11/dist-packages/bitsandbytes/cuda_setup/main.py:166: UserWarning: Welcome to bitsandbytes. For bug reports, please run

python -m bitsandbytes


  warn(msg)
The following directories listed in your path were found to be non-existent: {PosixPath('/usr/local/nvidia/lib'), PosixPath('/usr/local/lib/python3.11/dist-packages/cv2/../../lib64')}
/usr/local/lib/python3.11/dist-packages/bitsandbytes/cuda_setup/main.py:166: UserWarning: /usr/local/lib/python3.11/dist-packages/cv2/../../lib64:/usr/local/nvidia/lib:/usr/local/nvidia/lib64 did not contain ['libcudart.so', 'libcudart.so.11.0', 'libcudart.so.12.0'] as expected! Searching further paths...
  warn(msg)
The following directories listed in your path were found to be non-existent: {PosixPath('https'), PosixPath('//www.kaggle.com')}
The following directories listed in your path were found to be non-existent: {PosixPath('320043e14c

In [6]:
!nvidia-smi

Thu Aug 21 10:42:04 2025       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 560.35.03              Driver Version: 560.35.03      CUDA Version: 12.6     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   44C    P8             11W /   70W |       3MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----